# 04. Pembobotan Kata dengan TF-IDF

Notebook ini khusus untuk proses **Term Frequency - Inverse Document Frequency (TF-IDF)** menggunakan hasil teks bersih dari `dataset_preprocessing.csv`.

### Konsep TF-IDF:
- **TF (Term Frequency)**: Mengukur seberapa sering kata muncul dalam suatu dokumen.
- **IDF (Inverse Document Frequency)**: Mengukur seberapa penting atau langkanya kata tersebut pada seluruh kumpulan dokumen training:
  $$\text{IDF}(t) = \log\left(\frac{1 + N}{1 + \text{DF}(t)}\right) + 1$$
- **TF-IDF**: $\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$, dinormalisasi dengan L2-norm.

## 1. Import Library

In [1]:
import numpy as np
import pandas as pd
from collections import Counter
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


## 2. Membaca Dataset Preprocessing

In [2]:
df = pd.read_csv("dataset_preprocessing.csv")
print("Jumlah dokumen:", len(df))

# Tokenisasi per dokumen
dokumen_tokens = [str(text).split() for text in df["berita_clean"]]

Jumlah dokumen: 200


## 3. Train-test split
Split dilakukan sebelum fitting TF-IDF. `random_state=42` dan stratifikasi yang sama akan digunakan agar eksperimen TF-IDF dan Skip-gram memakai dokumen train/test yang sama.


In [3]:
LABEL_MAP = {"sport": 1, "finance": 2}
indeks_train, indeks_test = train_test_split(
    df.index,
    test_size=40,
    random_state=42,
    stratify=df["label"]
)
indeks_train = indeks_train.tolist()
indeks_test = indeks_test.tolist()

print(f"Jumlah Training: {len(indeks_train)} artikel")
print(f"Jumlah Testing : {len(indeks_test)} artikel")
print("Distribusi train:")
print(df.loc[indeks_train, "label"].value_counts().to_dict())
print("Distribusi test:")
print(df.loc[indeks_test, "label"].value_counts().to_dict())


Jumlah Training: 160 artikel
Jumlah Testing : 40 artikel
Distribusi train:
{'finance': 80, 'sport': 80}
Distribusi test:
{'finance': 20, 'sport': 20}


## 4. Pembentukan Vocabulary dan Nilai IDF

In [4]:
# Kosakata unik dari data training
vocabulary = sorted({
    kata for idx in indeks_train
    for kata in dokumen_tokens[idx]
    if kata != "label"
})

kata_to_idx = {kata: i for i, kata in enumerate(vocabulary)}

N = len(indeks_train)

# Hitung Document Frequency (DF)
df_count = np.zeros(len(vocabulary))

for idx in indeks_train:
    kata_unik_dokumen = set(dokumen_tokens[idx])

    for kata in kata_unik_dokumen:
        if kata in kata_to_idx:
            df_count[kata_to_idx[kata]] += 1

# Hitung IDF
idf = np.log((1 + N) / (1 + df_count)) + 1

print("Jumlah Vocabulary (Fitur Kata):", len(vocabulary))
print("Contoh 10 kata pertama:", vocabulary[:10])

Jumlah Vocabulary (Fitur Kata): 6308
Contoh 10 kata pertama: ['aaa', 'aau', 'abdullah', 'abidin', 'abimanyu', 'abk', 'abraham', 'abrian', 'absen', 'absennya']


## 5. Menghitung matriks TF-IDF
Vocabulary dan IDF sudah dipelajari dari data train. Fungsi berikut hanya memakai vocabulary dan IDF tersebut untuk mengubah dokumen menjadi vector.


In [5]:
def hitung_vektor_tfidf(tokens):
    counts = Counter(tokens)
    total_kata = len(tokens)
    vektor = np.zeros(len(vocabulary))

    if total_kata > 0:
        for kata, freq in counts.items():
            if kata in kata_to_idx:
                posisi = kata_to_idx[kata]
                tf = freq / total_kata
                vektor[posisi] = tf * idf[posisi]

    norm = np.linalg.norm(vektor)
    return vektor / norm if norm > 0 else vektor

X_train_tfidf = np.vstack([hitung_vektor_tfidf(dokumen_tokens[idx]) for idx in indeks_train])
X_test_tfidf = np.vstack([hitung_vektor_tfidf(dokumen_tokens[idx]) for idx in indeks_test])
y_train = df.loc[indeks_train, "label"].map(LABEL_MAP).to_numpy()
y_test = df.loc[indeks_test, "label"].map(LABEL_MAP).to_numpy()

print("Bentuk X_train_tfidf:", X_train_tfidf.shape)
print("Bentuk X_test_tfidf :", X_test_tfidf.shape)
print("Bentuk document-term matrix train:", X_train_tfidf.shape)
print("Contoh 10 fitur:", vocabulary[:10])


Bentuk X_train_tfidf: (160, 6308)
Bentuk X_test_tfidf : (40, 6308)
Bentuk document-term matrix train: (160, 6308)
Contoh 10 fitur: ['aaa', 'aau', 'abdullah', 'abidin', 'abimanyu', 'abk', 'abraham', 'abrian', 'absen', 'absennya']


## 6. Contoh hasil TF-IDF dan penyimpanan
Setiap baris matrix adalah dokumen, sedangkan setiap kolom adalah kata. Nilai yang lebih besar menunjukkan kata yang lebih penting dalam dokumen relatif terhadap data training.


In [6]:
print("label ada di vocabulary:", "label" in vocabulary)
print("id ada di vocabulary:", "id" in vocabulary)
print("Jumlah vocabulary:", len(vocabulary))

label ada di vocabulary: False
id ada di vocabulary: False
Jumlah vocabulary: 6308


In [7]:
df_tfidf = pd.DataFrame(np.vstack([X_train_tfidf, X_test_tfidf]), columns=vocabulary)
df_tfidf.insert(0, "split", ["train"] * len(X_train_tfidf) + ["test"] * len(X_test_tfidf))
df_tfidf.insert(0, "label", pd.concat([df.loc[indeks_train, "label"], df.loc[indeks_test, "label"]]).map(LABEL_MAP).to_numpy())
df_tfidf.insert(0, "id", pd.concat([df.loc[indeks_train, "id"], df.loc[indeks_test, "id"]]).to_numpy())

print("Ukuran DataFrame TF-IDF:", df_tfidf.shape)
display(df_tfidf.iloc[:5, :12])
df_tfidf.to_csv("hasil_tfidf.csv", index=False)
print("Data TF-IDF berhasil disimpan ke hasil_tfidf.csv")


Ukuran DataFrame TF-IDF: (200, 6311)


,id,label,split,aaa,aau,abdullah,abidin,abimanyu,abk,abraham,abrian,absen
0,FN055,2,train,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
1,SP076,1,train,0.0,0.0,0.033602,0.0,0.0,0.0,0.0,0.0,0.0
2,FN079,2,train,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
3,SP001,1,train,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
4,SP059,1,train,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0


Data TF-IDF berhasil disimpan ke hasil_tfidf.csv


## 7. Klasifikasi TF-IDF
TF-IDF menghasilkan fitur numerik. Tiga classifier dibandingkan pada split yang sama: Multinomial Naive Bayes, Logistic Regression, dan Linear SVM.


In [8]:
classifiers = {
    "TF-IDF + Naive Bayes": MultinomialNB(),
    "TF-IDF + Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "TF-IDF + SVM": LinearSVC(random_state=42)
}

hasil_klasifikasi = []
for nama, model in classifiers.items():
    model.fit(X_train_tfidf, y_train)
    prediksi = model.predict(X_test_tfidf)
    hasil_klasifikasi.append({
        "Model": nama,
        "Accuracy": accuracy_score(y_test, prediksi),
        "Precision_macro": precision_score(y_test, prediksi, average="macro", zero_division=0),
        "Recall_macro": recall_score(y_test, prediksi, average="macro", zero_division=0),
        "F1_macro": f1_score(y_test, prediksi, average="macro", zero_division=0)
    })

hasil_klasifikasi_tfidf = pd.DataFrame(hasil_klasifikasi).sort_values("F1_macro", ascending=False)
display(hasil_klasifikasi_tfidf)


,Model,Accuracy,Precision_macro,Recall_macro,F1_macro
0,TF-IDF + Naive Bayes,1.00,1.000000,1.00,1.000000
2,TF-IDF + SVM,1.00,1.000000,1.00,1.000000
1,TF-IDF + Logistic Regression,0.95,0.954545,0.95,0.949875


## 8. Interpretasi baseline TF-IDF
Semua classifier dilatih hanya menggunakan `X_train_tfidf`. Data test hanya digunakan untuk transformasi dengan vocabulary dan IDF train, lalu evaluasi. Dengan demikian, informasi label dan statistik test tidak digunakan saat fitting model. Macro F1 digunakan bersama accuracy agar performa kedua kelas dinilai seimbang.


## 9. Interpretasi fitur TF-IDF
Untuk memahami matrix, kita tampilkan kata dengan bobot TF-IDF terbesar pada satu dokumen training. Kata tersebut paling menonjol pada dokumen itu berdasarkan frekuensi dan kelangkaannya di data training.


In [9]:
contoh_dokumen = X_train_tfidf[0]
peringkat = np.argsort(contoh_dokumen)[::-1][:10]
top_kata = pd.DataFrame({"Kata": [vocabulary[i] for i in peringkat], "Nilai_TFIDF": contoh_dokumen[peringkat]})
display(top_kata)
hasil_klasifikasi_tfidf.to_csv("evaluasi_tfidf_classifier.csv", index=False)
print("Evaluasi classifier disimpan ke evaluasi_tfidf_classifier.csv")


,Kata,Nilai_TFIDF
0,vktr,0.326719
1,saham,0.262407
2,gems,0.224831
3,usd,0.223895
4,laba,0.192712
5,sema,0.192712
6,kontrak,0.167922
7,buy,0.160593
8,listrik,0.150068
9,miliar,0.150068


Evaluasi classifier disimpan ke evaluasi_tfidf_classifier.csv


## Audit STEP 1 - TF-IDF

### Tujuan audit
Memeriksa preprocessing, tokenisasi, pembentukan vocabulary, perhitungan TF-IDF, train-test split, dan potensi data leakage sebelum notebook diperbaiki.

### Temuan
- **Preprocessing dan tokenisasi:** notebook menggunakan kolom `berita_clean` dan tokenisasi `split()`. Ini dapat digunakan selama hasil preprocessing tersebut konsisten dengan notebook Skip-gram.
- **Train-test split:** pembagian data dilakukan manual berdasarkan 80 dokumen pertama setiap label untuk training. Jumlahnya 160 training dan 40 testing, tetapi split ini berbeda dari notebook Skip-gram yang menggunakan `train_test_split(random_state=42, stratify=...)`. Split perlu disamakan agar perbandingan adil.
- **Vocabulary dan IDF:** vocabulary, document frequency, dan IDF dibangun hanya dari indeks training. Ini sudah benar untuk mencegah informasi statistik data testing masuk ke proses fitting.
- **TF-IDF:** perhitungan `TF = frekuensi kata / jumlah kata`, `TF-IDF = TF * IDF`, smoothing IDF, dan normalisasi L2 sudah benar secara konsep.
- **Bentuk representasi:** hasil memiliki bentuk **Dokumen x Vocabulary**, yaitu `(200, 6269)`. Setiap baris adalah satu dokumen dan setiap kolom adalah satu kata/fitur.
- **Potensi leakage:** vocabulary dan IDF tidak dibangun dari data testing, sehingga leakage belum terjadi pada bagian tersebut. Namun, matrix saat ini dihitung untuk seluruh dokumen dan belum dipisahkan secara eksplisit menjadi `X_train_tfidf` dan `X_test_tfidf`. Ini perlu diperjelas sebelum klasifikasi.
- **Klasifikasi:** notebook baru menghasilkan fitur TF-IDF dan belum melakukan perbandingan classifier atau evaluasi.

### Kesimpulan audit
Fondasi perhitungan TF-IDF sudah benar dan tidak perlu diganti seluruhnya. Perbaikan berikutnya diperlukan untuk menyamakan split, memisahkan train/test secara eksplisit, menampilkan ukuran matrix dan contoh nilai TF-IDF, lalu menambahkan klasifikasi dan evaluasi.
